In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, log_loss
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print('imports OK')

---
## Problem 4: Session-Aware Sequential Recommendation (SASRec)

### Business Context
User browsing session shows real-time intent that differs from long-term profile. Capture in-session drift to improve recommendations. Target: <100ms SLA, HR@10 ≥ 0.35.

### Architecture
```
Session history: [item_3, item_7, item_1, item_9, ...]  (time-ordered)
                            |
                    Item Embeddings
                            |
             Self-Attention with Causal Mask
             (each position attends only to past)
                            |
                  Last Position Embedding
                            |
               dot(last_emb, candidate_emb)
                            |
                     Ranked candidates
```

### Key Concepts

**1. Causal Masking:** Position i can only attend to positions j ≤ i (no future peeking). Critical for sequential models — without it, the model sees future items during training, leaking information it won't have at inference.

**2. Why Transformers > RNNs for Session Modeling:**
- Attention is parallelizable (LSTM is sequential)
- Direct attention to any position, not just recent — handles long-range dependencies
- Multi-head captures multiple aspects of user interest simultaneously

**3. Cold-Start Graceful Degradation:**
```
if len(history) == 0:  popularity fallback (global top-k)
elif len(history) < 3: item-based CF (nearest neighbors to last item)
else:                  full SASRec pipeline
```

**4. Training: Binary Cross-Entropy with Negative Sampling**  
For each (session, next_item) pair: 1 positive + 100 random negatives. Loss penalizes ranking negatives above the positive.

In [ ]:
class SASRec(nn.Module):
    """
    Self-Attentive Sequential Recommendation.
    Simplified single-layer version for illustration.
    """
    def __init__(self, n_items, d_model=64, n_heads=2, max_seq_len=50, dropout=0.1):
        super().__init__()
        self.item_emb   = nn.Embedding(n_items + 1, d_model, padding_idx=0)
        self.pos_emb    = nn.Embedding(max_seq_len, d_model)
        self.attn       = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm       = nn.LayerNorm(d_model)
        self.dropout    = nn.Dropout(dropout)
        self.max_seq    = max_seq_len

    def forward(self, item_seq):
        """
        item_seq: (B, T) tensor of item ids (0 = padding)
        Returns: (B, T, d_model) contextual embeddings
        """
        B, T = item_seq.shape
        positions = torch.arange(T, device=item_seq.device).unsqueeze(0)  # (1, T)

        x = self.item_emb(item_seq) + self.pos_emb(positions)  # (B, T, d)
        x = self.dropout(x)

        # Causal mask: position i cannot attend to j > i
        causal_mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()

        # Padding mask: ignore padded positions (item_id=0)
        key_padding_mask = (item_seq == 0)  # (B, T)

        attn_out, _ = self.attn(x, x, x,
                                attn_mask=causal_mask,
                                key_padding_mask=key_padding_mask)
        x = self.norm(x + attn_out)   # residual + layer norm
        return x

    def score_candidates(self, item_seq, candidate_ids):
        """
        Score candidate items given session history.
        candidate_ids: (B, K) candidate item ids
        Returns: (B, K) scores
        """
        seq_emb = self.forward(item_seq)          # (B, T, d)
        last    = seq_emb[:, -1, :]               # (B, d) — use last position
        cand    = self.item_emb(candidate_ids)    # (B, K, d)
        scores  = torch.bmm(cand, last.unsqueeze(-1)).squeeze(-1)  # (B, K)
        return scores

# --- Example usage ---
n_items, B, T, K = 500, 8, 20, 10
model = SASRec(n_items=n_items, d_model=64, n_heads=2, max_seq_len=T)

# Simulated session sequences (0 = padding)
item_seq = torch.randint(1, n_items + 1, (B, T))
item_seq[:, :5] = 0   # first 5 positions are padding

# Candidate items to score
candidates = torch.randint(1, n_items + 1, (B, K))

scores = model.score_candidates(item_seq, candidates)
top1 = scores.argmax(dim=-1)
print(f'Scores shape: {scores.shape}')   # (B, K)
print(f'Top-1 candidate per user: {top1.tolist()}')

# --- Hit Rate@K metric ---
def hit_rate_at_k(scores, ground_truth_idx, k):
    """Fraction of users where the true next item is in top-k."""
    top_k = scores.topk(k, dim=-1).indices  # (B, k)
    hits = (top_k == ground_truth_idx.unsqueeze(-1)).any(dim=-1)
    return hits.float().mean().item()

ground_truth = torch.randint(0, K, (B,))   # index of the true next item in candidates
print(f'\nHR@1: {hit_rate_at_k(scores, ground_truth, 1):.3f}')
print(f'HR@5: {hit_rate_at_k(scores, ground_truth, 5):.3f}')

### L6 Interview Q&A

**Q: Why is the causal mask essential in SASRec?**  
A: Without it, position i can attend to future items j > i. During training, the model sees the answer (future items in the same session), learning a trivial shortcut. At inference, future items don't exist — the model is in a different regime. The causal mask ensures the model only uses past context, making training and inference consistent.

**Q: User's session has only 2 items. What do you do?**  
A: Fall back to item-based CF — find nearest neighbors to the last item by embedding similarity. If 0 items, fall back to global popularity. These fallbacks are critical for cold-start and must be implemented explicitly — never feed a nearly-empty sequence to SASRec as it will produce unreliable embeddings.

**Q: How do you evaluate the session model offline?**  
A: Leave-last-out evaluation: for each session, hold out the last item, train on all others. Measure HR@10 (was true next item in top-10?) and NDCG@10 (how highly was it ranked?). Use chronological splitting — never shuffle sessions randomly, as that leaks future data into training.

### Common Pitfalls
- Shuffling data randomly instead of chronological train/test split — data leakage
- Missing the causal mask — model won't generalize to inference
- Not handling padding correctly in attention (key_padding_mask)